# Phase 3 : Modélisation

Ce notebook charge les données brutes et le pipeline de prétraitement pour entamer la modélisation.

**Étapes :**
1. Chargement de `dataset.csv`
2. Séparation des features `X` et de la cible `y` (`abandoned`)
3. Split stratifié en Train / Validation / Test (70/15/15)
4. Chargement du pipeline `preprocessor.joblib`
5. Vérification de la distribution de la variable cible pour confirmer le déséquilibre.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# 1. Chargement des données brutes
df = pd.read_csv('../data/dataset.csv')

# 2. Séparation des features (X) et de la cible (y)
X = df.drop(columns=['abandoned'])
y = df['abandoned']

# 3. Train / Validation / Test split (70/15/15) stratifié
# On sépare d'abord 30% pour un set temporaire (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Puis on divise le set temporaire en deux parties égales (50% Val, 50% Test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# 4. Chargement du pipeline de prétraitement
# Assurez-vous d'avoir scikit-learn d'installé dans l'environnement courant
preprocessor = joblib.load('../models/preprocessor.joblib')

# 5. Affichage des shapes et de la distribution
print("Shapes des sous-ensembles :")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

print("\nDistribution de y_train (%) :")
print((y_train.value_counts(normalize=True) * 100).round(2))

### 2.1 Baseline — Régression Logistique

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

pipeline_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1', 'recall', 'precision']

results_lr = cross_validate(pipeline_lr, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Régression Logistique :")
print(f"F1-score  : {np.mean(results_lr['test_f1']):.3f} ± {np.std(results_lr['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_lr['test_recall']):.3f} ± {np.std(results_lr['test_recall']):.3f}")
print(f"Precision : {np.mean(results_lr['test_precision']):.3f} ± {np.std(results_lr['test_precision']):.3f}")


### 2.2 Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

pipeline_dt = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

results_dt = cross_validate(pipeline_dt, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Decision Tree :")
print(f"F1-score  : {np.mean(results_dt['test_f1']):.3f} ± {np.std(results_dt['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_dt['test_recall']):.3f} ± {np.std(results_dt['test_recall']):.3f}")
print(f"Precision : {np.mean(results_dt['test_precision']):.3f} ± {np.std(results_dt['test_precision']):.3f}")


### 2.3 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced_subsample', n_estimators=200, random_state=42))
])

results_rf = cross_validate(pipeline_rf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Random Forest :")
print(f"F1-score  : {np.mean(results_rf['test_f1']):.3f} ± {np.std(results_rf['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_rf['test_recall']):.3f} ± {np.std(results_rf['test_recall']):.3f}")
print(f"Precision : {np.mean(results_rf['test_precision']):.3f} ± {np.std(results_rf['test_precision']):.3f}")

### 2.4 XGBoost

In [ ]:
from xgboost import XGBClassifier

scale = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(scale_pos_weight=scale, eval_metric='logloss', random_state=42))
])

results_xgb = cross_validate(pipeline_xgb, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("XGBoost :")
print(f"F1-score  : {np.mean(results_xgb['test_f1']):.3f} ± {np.std(results_xgb['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_xgb['test_recall']):.3f} ± {np.std(results_xgb['test_recall']):.3f}")
print(f"Precision : {np.mean(results_xgb['test_precision']):.3f} ± {np.std(results_xgb['test_precision']):.3f}")

### 2.5 MLP

In [ ]:
from sklearn.neural_network import MLPClassifier

pipeline_mlp = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))
])

results_mlp = cross_validate(pipeline_mlp, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("MLP :")
print(f"F1-score  : {np.mean(results_mlp['test_f1']):.3f} ± {np.std(results_mlp['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_mlp['test_recall']):.3f} ± {np.std(results_mlp['test_recall']):.3f}")
print(f"Precision : {np.mean(results_mlp['test_precision']):.3f} ± {np.std(results_mlp['test_precision']):.3f}")

### 2.6 Tableau comparatif des 5 modeles (sans reequilibrage)

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    'Modele': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'MLP'],
    'F1 mean': [np.mean(results_lr['test_f1']), np.mean(results_dt['test_f1']), np.mean(results_rf['test_f1']), np.mean(results_xgb['test_f1']), np.mean(results_mlp['test_f1'])],
    'F1 std': [np.std(results_lr['test_f1']), np.std(results_dt['test_f1']), np.std(results_rf['test_f1']), np.std(results_xgb['test_f1']), np.std(results_mlp['test_f1'])],
    'Recall mean': [np.mean(results_lr['test_recall']), np.mean(results_dt['test_recall']), np.mean(results_rf['test_recall']), np.mean(results_xgb['test_recall']), np.mean(results_mlp['test_recall'])],
    'Precision mean': [np.mean(results_lr['test_precision']), np.mean(results_dt['test_precision']), np.mean(results_rf['test_precision']), np.mean(results_xgb['test_precision']), np.mean(results_mlp['test_precision'])]
}).sort_values('F1 mean', ascending=False).round(3)

display(summary)

print("Meilleur modele baseline :", summary.iloc[0]['Modele'])

### 2.7 SMOTE sur tous les modeles

In [ ]:
from imblearn.over_sampling import SMOTE

pipelines_smote = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', LogisticRegression(max_iter=5000, random_state=42))]),
    'Decision Tree': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', DecisionTreeClassifier(random_state=42))]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'XGBoost': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', XGBClassifier(eval_metric='logloss', random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('smote', SMOTE(random_state=42)), ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))])
}

results_smote = {}
for name, pipe in pipelines_smote.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results_smote[name] = res
    print(f"{name} (SMOTE) :")
    print(f"  F1-score  : {np.mean(res['test_f1']):.3f} +/- {np.std(res['test_f1']):.3f}")
    print(f"  Recall    : {np.mean(res['test_recall']):.3f} +/- {np.std(res['test_recall']):.3f}")
    print(f"  Precision : {np.mean(res['test_precision']):.3f} +/- {np.std(res['test_precision']):.3f}")

### 2.8 RandomUnderSampler sur tous les modeles

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

pipelines_rus = {
    'Logistic Regression': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', LogisticRegression(max_iter=5000, random_state=42))]),
    'Decision Tree': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', DecisionTreeClassifier(random_state=42))]),
    'Random Forest': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', RandomForestClassifier(n_estimators=200, random_state=42))]),
    'XGBoost': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', XGBClassifier(eval_metric='logloss', random_state=42))]),
    'MLP': Pipeline([('preprocessor', preprocessor), ('rus', RandomUnderSampler(random_state=42)), ('model', MLPClassifier(early_stopping=True, max_iter=500, random_state=42))])
}

results_rus = {}
for name, pipe in pipelines_rus.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    results_rus[name] = res
    print(f"{name} (RandomUnderSampler) :")
    print(f"  F1-score  : {np.mean(res['test_f1']):.3f} +/- {np.std(res['test_f1']):.3f}")
    print(f"  Recall    : {np.mean(res['test_recall']):.3f} +/- {np.std(res['test_recall']):.3f}")
    print(f"  Precision : {np.mean(res['test_precision']):.3f} +/- {np.std(res['test_precision']):.3f}")

### 2.9 Tableau comparatif 5x3

In [ ]:
modeles = ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'MLP']

data = []
for name in modeles:
    data.append({
        'Modele': name,
        'Sans reeq. F1': round(np.mean(results_lr['test_f1'] if name=='Logistic Regression' else results_dt['test_f1'] if name=='Decision Tree' else results_rf['test_f1'] if name=='Random Forest' else results_xgb['test_f1'] if name=='XGBoost' else results_mlp['test_f1']), 3),
        'SMOTE F1': round(np.mean(results_smote[name]['test_f1']), 3),
        'RUS F1': round(np.mean(results_rus[name]['test_f1']), 3),
        'Sans reeq. Recall': round(np.mean(results_lr['test_recall'] if name=='Logistic Regression' else results_dt['test_recall'] if name=='Decision Tree' else results_rf['test_recall'] if name=='Random Forest' else results_xgb['test_recall'] if name=='XGBoost' else results_mlp['test_recall']), 3),
        'SMOTE Recall': round(np.mean(results_smote[name]['test_recall']), 3),
        'RUS Recall': round(np.mean(results_rus[name]['test_recall']), 3),
    })

df_comparatif = pd.DataFrame(data)
display(df_comparatif)

### 2.10 Selection de la meilleure combinaison

In [ ]:
results_map = {
    'Logistic Regression': results_lr,
    'Decision Tree': results_dt,
    'Random Forest': results_rf,
    'XGBoost': results_xgb,
    'MLP': results_mlp
}

strategies = {
    'Sans reeq.': results_map,
    'SMOTE': results_smote,
    'RUS': results_rus
}

best_f1 = 0
best_recall = 0
best_combo = ('', '')

for strat_name, strat_results in strategies.items():
    for model_name in modeles:
        f1 = np.mean(strat_results[model_name]['test_f1'])
        recall = np.mean(strat_results[model_name]['test_recall'])
        precision = np.mean(strat_results[model_name]['test_precision'])
        if recall >= 0.80 and f1 > best_f1:
            best_f1 = f1
            best_recall = recall
            best_combo = (model_name, strat_name)

if best_combo == ('', ''):
    print("Aucune combinaison n'atteint recall >= 0.80")
    print("Selection par meilleur F1 global :")
    for strat_name, strat_results in strategies.items():
        for model_name in modeles:
            f1 = np.mean(strat_results[model_name]['test_f1'])
            if f1 > best_f1:
                best_f1 = f1
                best_combo = (model_name, strat_name)

best_model_name, best_strategy = best_combo
print(f"Combinaison retenue pour le tuning : {best_model_name} + {best_strategy}")
print(f"F1 : {best_f1:.3f} | Recall : {best_recall:.3f}")

### 2.11 Evaluation de la meilleure combinaison sur validation

In [ ]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, classification_report, confusion_matrix

pipelines_all = {
    'Sans reeq.': {
        'Logistic Regression': pipeline_lr,
        'Decision Tree': pipeline_dt,
        'Random Forest': pipeline_rf,
        'XGBoost': pipeline_xgb,
        'MLP': pipeline_mlp
    },
    'SMOTE': pipelines_smote,
    'RUS': pipelines_rus
}

best_pipe = pipelines_all[best_strategy][best_model_name]
best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_val)

print(f"Evaluation sur X_val — {best_model_name} + {best_strategy}")
print(f"F1        : {f1_score(y_val, y_pred):.3f}  (seuil : >= 0.55)")
print(f"Recall    : {recall_score(y_val, y_pred):.3f}  (seuil : >= 0.80)")
print(f"Precision : {precision_score(y_val, y_pred):.3f}  (seuil : >= 0.40)")
print(f"Accuracy  : {accuracy_score(y_val, y_pred):.3f}")
print()
print(classification_report(y_val, y_pred, target_names=['Complete', 'Abandonne']))
print()
print("Matrice de confusion :")
print(confusion_matrix(y_val, y_pred))

### Synthese finale

## Synthese

### Modeles testes
5 modeles ont ete evalues : Logistic Regression (baseline), Decision Tree, Random Forest, XGBoost et MLP.

### Strategies de desequilibre comparees
3 strategies ont ete comparees pour chaque modele :
- Sans reeequilibrage (class_weight/scale_pos_weight)
- SMOTE (oversampling de la minorite)
- RandomUnderSampler (undersampling de la majorite)

### Tableau 5x3
15 combinaisons evaluees par validation croisee stratifiee (5 folds), avec F1, Recall et Precision reportes pour chaque combinaison.

### Combinaison retenue
La meilleure combinaison est selectionnee en priorite sur recall >= 0.80, puis sur F1 maximal.

### Objectifs metier
- OM-1 : Recall >= 0.80 (detecter 80% des essais abandonnes) ? cout FN : 50-100M$
- OM-2 : Precision >= 0.40 (limiter les fausses alertes) ? cout FP : 50K-200K$
- OM-3 : F1 >= 0.55 (equilibre global)

### Suite
La combinaison retenue sera tunee dans le notebook 05_tuning.ipynb.